Cell 1: Imports and Device Setup

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# fixme: always sets the device to CPU
# Check for GPU availability to drastically speed up training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

Executing on device: cpu


Cell 2: Custom Activation Function

In [2]:
class TriangularActivation(nn.Module):
    """
    Implements the triangular activation function used in the paper's hidden layers.
    Mathematically equivalent to MATLAB's 'tribas': f(x) = max(1 - |x|, 0)
    """
    def forward(self, x):
        return torch.clamp(1.0 - torch.abs(x), min=0.0)

Cell 3: Highly Optimized Dataset Class

In [3]:
class OFDMDataset(Dataset):
    def __init__(self, file_list, part='real'):
        self.file_list = file_list
        self.part = part

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        # 1. Load the specific .npz file
        file_path = self.file_list[idx]
        data = np.load(file_path)

        # Extract input (original OFDM) and target (ICF) data
        X_raw = data['tx']
        Y_raw = data['rx']

        # 2. Extract Real (0) or Imaginary (1) part
        part_idx = 0 if self.part == 'real' else 1
        X_batch = torch.tensor(X_raw[part_idx, :, :], dtype=torch.float32)
        Y_batch = torch.tensor(Y_raw[part_idx, :, :], dtype=torch.float32)

        # 3. Native PyTorch Normalization to strictly [-1, 1]
        X_min, X_max = X_batch.min(), X_batch.max()
        Y_min, Y_max = Y_batch.min(), Y_batch.max()

        # Avoid division by zero in case of an empty/flat signal
        if (X_max - X_min) != 0:
            X_norm = 2.0 * ((X_batch - X_min) / (X_max - X_min)) - 1.0
        else:
            X_norm = X_batch

        if (Y_max - Y_min) != 0:
            Y_norm = 2.0 * ((Y_batch - Y_min) / (Y_max - Y_min)) - 1.0
        else:
            Y_norm = Y_batch

        return X_norm, Y_norm

Cell 4: Neural Network Architecture

In [4]:
class NNICFMapper(nn.Module):
    def __init__(self, input_size):
        super(NNICFMapper, self).__init__()

        # First hidden layer: 2 neurons
        self.hidden1 = nn.Linear(input_size, 2)

        # Second hidden layer: 1 neuron
        self.hidden2 = nn.Linear(2, 1)

        # Output layer: Maps back to the original subcarrier points
        self.output = nn.Linear(1, input_size)

        # Custom triangular activation
        self.activation = TriangularActivation()

    def forward(self, x):
        x = self.activation(self.hidden1(x))
        x = self.activation(self.hidden2(x))
        x = self.output(x) # Standard linear output
        return x

Cell 5: Initialization and DataLoaders

In [5]:
# --- Configuration ---
# Update this path to where your 100 files are saved!
file_directory = "./16QAMdata/"
all_files = [os.path.join(file_directory, f"16qam_tx_rx_32_part_{i:03d}.npz") for i in range(100)]

# Initialize DataLoaders with asynchronous loading (num_workers) and rapid memory transfer
train_dataset_real = OFDMDataset(all_files, part='real')
train_loader_real = DataLoader(
    train_dataset_real,
    batch_size=None, # batch_size is None because the dataset returns a full batch of 10,000
    shuffle=True,
    # num_workers=2,   # Adjust between 2-4 based on your CPU cores
    # pin_memory=True
)

train_dataset_imag = OFDMDataset(all_files, part='imag')
train_loader_imag = DataLoader(
    train_dataset_imag,
    batch_size=None,
    shuffle=True,
    # num_workers=2,
    # pin_memory=True
)

# 1024 features based on N=256 and oversampling J=4
input_features = 1024

# Initialize models and send them to the GPU/device
Mod_Re_NN = NNICFMapper(input_features).to(device)
Mod_Im_NN = NNICFMapper(input_features).to(device)

# Standard Mean Squared Error loss and Adam optimizer
criterion = nn.MSELoss()
optimizer_real = optim.Adam(Mod_Re_NN.parameters(), lr=0.001)
optimizer_imag = optim.Adam(Mod_Im_NN.parameters(), lr=0.001)

Cell 6: Training Loop (Real Module)

In [6]:
# fixme: took ~2h50m for 80 epochs
epochs = 100
print("--- Starting Training for the Real NN Module ---")

for epoch in range(epochs):
    epoch_loss = 0.0
    Mod_Re_NN.train()

    for X_batch, Y_batch in train_loader_real:
        # Transfer data to GPU
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        optimizer_real.zero_grad()
        predictions = Mod_Re_NN(X_batch)
        loss = criterion(predictions, Y_batch)

        loss.backward()
        optimizer_real.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_real)
    print(f"Real Module | Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.6f}")

print("Real NN Module Training Complete!")

--- Starting Training for the Real NN Module ---
Real Module | Epoch 1/100 | Loss: 0.359762
Real Module | Epoch 2/100 | Loss: 0.267390
Real Module | Epoch 3/100 | Loss: 0.194103
Real Module | Epoch 4/100 | Loss: 0.143748
Real Module | Epoch 5/100 | Loss: 0.112743
Real Module | Epoch 6/100 | Loss: 0.094719
Real Module | Epoch 7/100 | Loss: 0.084615
Real Module | Epoch 8/100 | Loss: 0.079135
Real Module | Epoch 9/100 | Loss: 0.076248
Real Module | Epoch 10/100 | Loss: 0.074783
Real Module | Epoch 11/100 | Loss: 0.074066
Real Module | Epoch 12/100 | Loss: 0.073726
Real Module | Epoch 13/100 | Loss: 0.073571
Real Module | Epoch 14/100 | Loss: 0.073506
Real Module | Epoch 15/100 | Loss: 0.073477
Real Module | Epoch 16/100 | Loss: 0.073465
Real Module | Epoch 17/100 | Loss: 0.073461
Real Module | Epoch 18/100 | Loss: 0.073459
Real Module | Epoch 19/100 | Loss: 0.073458
Real Module | Epoch 20/100 | Loss: 0.073461
Real Module | Epoch 21/100 | Loss: 0.073460
Real Module | Epoch 22/100 | Loss: 0

KeyboardInterrupt: 

Cell 7: Training Loop (Imaginary Module)

In [ ]:
print("--- Starting Training for the Imaginary NN Module ---")

for epoch in range(epochs):
    epoch_loss = 0.0
    Mod_Im_NN.train()

    for X_batch, Y_batch in train_loader_imag:
        # Transfer data to GPU
        X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

        optimizer_imag.zero_grad()
        predictions = Mod_Im_NN(X_batch)
        loss = criterion(predictions, Y_batch)

        loss.backward()
        optimizer_imag.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_imag)
    print(f"Imaginary Module | Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.6f}")

print("Imaginary NN Module Training Complete!")